## BU

Bottom-up **order reconciliation** using the cleaned simulation pipeline.

In [1]:
path1 = 'E:/yjz/Extension for hts/JayCode/Models/'

import warnings
warnings.simplefilter("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import BottomUp

from inventory_pipeline import save_pickle, run_fixed_loop

In [2]:
test = pd.read_pickle(f"{path1}721future_28.pkl").reset_index(drop=True)
ts_id = test[['unique_id', 'ds']].reset_index(drop=True)
truth = test['y'].reset_index(drop=True)

S = pd.read_pickle(f"{path1}721S_df.pkl")
tags = pd.read_pickle(f"{path1}tags.bin")

lead_time = 1
MODEL_NAMES = ['lgb_base', 'lgb_bu', 'lgb_td', 'lgb_mint',
               'ets_base', 'ets_bu', 'ets_td', 'ets_mint']
on = ['ot_90', 'ot_95', 'ot_99']

lgbs = pd.read_pickle(f"lgbInvtSim_L{lead_time}.pkl").reset_index(drop=True)
etss = pd.read_pickle(f"etsInvtSim_L{lead_time}.pkl").reset_index(drop=True)
base_all = pd.concat([lgbs, etss], ignore_index=True)

bu = HierarchicalReconciliation(reconcilers=[BottomUp()])

In [9]:
base_all.iloc[:28,:].to_csv('base_sample.csv')

In [3]:
lobu = []
for name in tqdm(MODEL_NAMES):
    sub_orders = base_all.loc[base_all['name'] == name, on].reset_index(drop=True)
    y_hat = pd.concat([ts_id, sub_orders], axis=1)
    rec = bu.reconcile(Y_hat_df=y_hat, S=S, tags=tags).iloc[:,-3:].copy()
    rec.insert(0, 'name', name)
    lobu.append(rec.reset_index(drop=True))

OrderBU = pd.concat(lobu, ignore_index=True)
save_pickle(OrderBU, f"OrderBU_L{lead_time}.pkl")
OrderBU.head()

100%|███████████████████████████████████████████████████████████████████████████████████| 8/8 [16:11<00:00, 121.44s/it]


,name,ot_90/BottomUp,ot_95/BottomUp,ot_99/BottomUp
0,lgb_base,15.747402,20.211572,28.585611
1,lgb_base,4.350416,4.350416,4.350416
2,lgb_base,4.451770,4.451770,4.451770
3,lgb_base,7.910317,7.910317,7.910317
4,lgb_base,1.017425,1.017425,1.017425


In [4]:
['name']+on

['name', 'ot_90', 'ot_95', 'ot_99']

In [5]:
OrderBU.columns = ['name']+on
OrderBU

,name,ot_90,ot_95,ot_99
0,lgb_base,15.747402,20.211572,28.585611
1,lgb_base,4.350416,4.350416,4.350416
2,lgb_base,4.451770,4.451770,4.451770
3,lgb_base,7.910317,7.910317,7.910317
4,lgb_base,1.017425,1.017425,1.017425
...,...,...,...,...
9561659,ets_mint,0.000000,0.000000,0.000000
9561660,ets_mint,0.000000,0.000000,0.000000
9561661,ets_mint,0.021772,0.021772,0.021772
9561662,ets_mint,0.025888,0.025888,0.025888


In [6]:
Invt_df = []
for name in MODEL_NAMES:
    base_ref = base_all.loc[base_all['name'] == name].reset_index(drop=True)
    order_ref = OrderBU.loc[OrderBU['name'] == name].reset_index(drop=True)

    fixed_orders = pd.concat(
        [
            base_ref[['forecasts', 'sst_90', 'sst_95', 'sst_99']].reset_index(drop=True),
            order_ref[['ot_90', 'ot_95', 'ot_99']].reset_index(drop=True),
        ],
        axis=1,
    )

    df = run_fixed_loop(
        truth=truth,
        forecast_source=base_ref['forecasts'].reset_index(drop=True),
        fixed_orders=fixed_orders,
        NAME=name,
        L_=lead_time,
    )
    Invt_df.append(df)

BUOrder = pd.concat(Invt_df, ignore_index=True)
save_pickle(BUOrder, f"BUOrder_L{lead_time}.pkl")
BUOrder.head()

100%|███████████████████████████████████████████████████████████████████████████| 42686/42686 [01:20<00:00, 532.82it/s]


,name,true_demand,forecasts,sst_90,arrival_90,ot_90,wip_90,ip_90,net_90,backlog_90,...,cb_95,sst_99,arrival_99,ot_99,wip_99,ip_99,net_99,backlog_99,ch_99,cb_99
0,lgb_base,4.0,5.689794,5.648312,0.000000,15.747402,15.747402,17.437196,1.689794,0.0,...,0.0,10.253148,0.000000,28.585611,28.585611,30.275405,1.689794,0.0,1.689794,0.0
1,lgb_base,5.0,4.679130,5.648312,15.747402,4.350416,4.350416,16.787612,12.437196,0.0,...,0.0,10.253148,28.585611,4.350416,4.350416,29.625822,25.275405,0.0,25.275405,0.0
2,lgb_base,7.0,4.798928,5.648312,4.350416,4.451770,4.451770,14.239382,9.787612,0.0,...,0.0,10.253148,4.350416,4.451770,4.451770,27.077592,22.625822,0.0,22.625822,0.0
3,lgb_base,1.0,5.980217,5.648312,4.451770,7.910317,7.910317,21.149699,13.239382,0.0,...,0.0,10.253148,4.451770,7.910317,7.910317,33.987908,26.077592,0.0,26.077592,0.0
4,lgb_base,9.0,5.048564,5.648312,7.910317,1.017425,1.017425,13.167124,12.149699,0.0,...,0.0,10.253148,7.910317,1.017425,1.017425,26.005333,24.987908,0.0,24.987908,0.0
